# Penguin Measurements EDA (Palmer Archipelago, Antarctica)
Basic exploratory data analysis using **Python, NumPy and Pandas** (no machine learning).

Dataset: `penguins_size.csv` (344 rows) - https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data

Place `penguins_size.csv` in the same folder as this file before running.

## Step 1: Import libraries and load the dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("penguins_size.csv")

# The file calls bill measurements "culmen"; rename them to "bill" for clarity
df = df.rename(columns={"culmen_length_mm": "bill_length_mm",
                        "culmen_depth_mm": "bill_depth_mm"})

# One sex value is written as "." (a placeholder). Treat it as missing.
df["sex"] = df["sex"].replace(".", np.nan)
df["sex"] = df["sex"].str.title()

## Step 2: Basic inspection - head(), tail(), shape, info(), describe()

In [ ]:
print("First 5 rows:")
print(df.head())

In [ ]:
print("Last 5 rows:")
print(df.tail())

In [ ]:
print("Shape (rows, columns):", df.shape)

In [ ]:
df.info()

In [ ]:
print(df.describe())                 # numeric columns
print()
print(df.describe(include="object")) # text columns

## Step 3: Check missing values and duplicates

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())

## Step 4: NumPy array calculations

In [ ]:
mass = df["body_mass_g"].to_numpy(dtype=float)           # NumPy array
flipper = df["flipper_length_mm"].to_numpy(dtype=float)  # NumPy array

print("Mean body mass   :", round(np.nanmean(mass), 1), "g")
print("Median body mass :", np.nanmedian(mass), "g")
print("Std dev          :", round(np.nanstd(mass), 1), "g")
print("Min / Max        :", np.nanmin(mass), "/", np.nanmax(mass), "g")

q1, q3 = np.nanpercentile(mass, [25, 75])
print("25th / 75th percentile:", q1, "/", q3)

# Correlation between flipper length and body mass (use rows where both exist)
both = ~np.isnan(mass) & ~np.isnan(flipper)
r = np.corrcoef(flipper[both], mass[both])[0, 1]
print("Correlation (flipper vs body mass): r =", round(r, 3))

## Question 1: Count penguins by species and island

In [ ]:
print("By species:")
print(df["species"].value_counts())
print()
print("By island:")
print(df["island"].value_counts())
print()
print("By species and island (groupby):")
print(df.groupby(["species", "island"]).size())
print()
print("As a table (crosstab):")
print(pd.crosstab(df["species"], df["island"], margins=True))

**Answer:** Adelie = 152 (Biscoe 44, Dream 56, Torgersen 52), Chinstrap = 68 (Dream only), Gentoo = 124 (Biscoe only). Total = 344.

## Question 2: Average body mass and flipper length by species

In [ ]:
avg_by_species = df.groupby("species")[["body_mass_g", "flipper_length_mm"]].mean().round(2)
print(avg_by_species)

**Answer:** Adelie 3700.66 g / 189.95 mm, Chinstrap 3733.09 g / 195.82 mm, Gentoo 5076.02 g / 217.19 mm.

## Question 3: Which species has the highest average body mass?

In [ ]:
heaviest = avg_by_species["body_mass_g"].idxmax()
print(heaviest, "has the highest average body mass:",
      avg_by_species["body_mass_g"].max(), "g")
print()
print(avg_by_species["body_mass_g"].sort_values(ascending=False))

**Answer:** **Gentoo**, with an average body mass of 5076.02 g.

## Question 4: Compare average bill length by species

In [ ]:
bill_by_species = df.groupby("species")["bill_length_mm"].mean().round(2).sort_values(ascending=False)
print(bill_by_species)

**Answer:** Chinstrap 48.83 mm (longest), Gentoo 47.50 mm, Adelie 38.79 mm (shortest).

## Question 5: Check and report missing values in sex and measurement columns

In [ ]:
check_cols = ["sex", "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
print(df[check_cols].isnull().sum())
print()
print("Percent missing:")
print((df[check_cols].isnull().mean() * 100).round(2))
print()
print("Rows where the measurements are missing:")
print(df[df[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]].isnull().any(axis=1)])
print()
print("Missing sex by species:")
print(df[df["sex"].isnull()].groupby("species").size())

**Answer:** `sex` has 11 missing values (3.2%). Each of the four measurement columns has 2 missing values (0.58%), and they are the same two rows (index 3 and 339). There are no duplicate rows.

## Question 6: Find the 10 penguins with the greatest body mass

In [ ]:
top10 = df.dropna(subset=["body_mass_g"]).sort_values("body_mass_g", ascending=False).head(10)
print(top10)
print()
print(top10["species"].value_counts())

**Answer:** All 10 heaviest penguins are male Gentoo from Biscoe Island, ranging from 5800 g to 6300 g.

## Extra: Pandas filtering example

In [ ]:
heavy = df[df["body_mass_g"] > 5000]
print("Penguins heavier than 5000 g:", len(heavy))
print(heavy["species"].value_counts())

## Chart 1: Species-count bar chart

In [ ]:
species_counts = df["species"].value_counts()

plt.figure(figsize=(7, 5))
bars = plt.bar(species_counts.index, species_counts.values,
               color=["#4C78A8", "#F58518", "#54A24B"])
plt.bar_label(bars)
plt.title("Number of Penguins by Species")
plt.xlabel("Species")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("species_count_bar.png", dpi=150)
plt.show()

## Chart 2: Body-mass histogram

In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(mass[~np.isnan(mass)], bins=20, color="#4C78A8", edgecolor="white")
plt.axvline(np.nanmean(mass), color="red", linestyle="--",
            label=f"Mean = {np.nanmean(mass):.0f} g")
plt.title("Distribution of Penguin Body Mass")
plt.xlabel("Body mass (g)")
plt.ylabel("Number of penguins")
plt.legend()
plt.tight_layout()
plt.savefig("body_mass_hist.png", dpi=150)
plt.show()

## Three Observations
1. **Species and islands:** Adelie is the most common species (152 of 344 penguins, 44.2%) and is the only species found on all three islands. Chinstrap is found only on Dream and Gentoo only on Biscoe.
2. **Body size:** Gentoo is the heaviest species (average 5076 g), about 1375 g heavier than Adelie (3701 g), and has the longest flippers (217 mm). Chinstrap has the longest bills (48.83 mm). Flipper length and body mass are strongly related (r = 0.87), and all 10 heaviest penguins are Gentoo.
3. **Data quality:** The `sex` column has the most missing values (11, versus 2 in each measurement column), and there are no duplicate rows. The mean body mass (4202 g) is higher than the median (4050 g), and the histogram shows more than one cluster, which reflects the mix of smaller (Adelie, Chinstrap) and larger (Gentoo) species.